# Naive Fine-tuning: Evaluación Class-IL

Este notebook entrena un baseline Naive en el escenario **Class-IL**. 
Aquí el modelo tiene una única cabeza de salida y debe decidir entre todas las clases vistas hasta ahora sin saber el ID de la tarea.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, ClassIncrementalClassifier
from dataloaders import SequentialCIFAR10
from utils_class_il import evaluate_class_il

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 10
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


In [2]:
seq_cifar = SequentialCIFAR10(batch_size=BATCH_SIZE)
backbone = CNN(in_channels=3, embedding_dim=32)
model = ClassIncrementalClassifier(backbone, embedding_dim=32, total_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for task_id in range(5):
    print(f"\n--- ENTRENANDO TAREA {task_id} ---")
    
    # 1. Activar nuevas clases en el modelo
    current_classes = seq_cifar.task_classes[task_id]
    model.add_task(current_classes)
    
    # 2. Obtener datos de la tarea actual
    train_ds = seq_cifar.get_task_train_dataset(task_id, remap_labels=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    # 3. Entrenamiento Naive
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")
    
    # 4. Evaluación Class-IL (todas las clases vistas hasta ahora)
    all_test = ConcatDataset([seq_cifar.get_task_test_dataset(tid) for tid in range(task_id + 1)])
    combined_loader = DataLoader(all_test, batch_size=BATCH_SIZE, shuffle=False)
    
    acc_class_il = evaluate_class_il(model, combined_loader, device, [])
    print(f"Precisión Class-IL tras Tarea {task_id}: {acc_class_il:.2f}%")


--- ENTRENANDO TAREA 0 ---
  Epoch 1/10 | Loss: 0.4713
  Epoch 2/10 | Loss: 0.3601
  Epoch 3/10 | Loss: 0.3147
  Epoch 4/10 | Loss: 0.2939
  Epoch 5/10 | Loss: 0.2883
  Epoch 6/10 | Loss: 0.2646
  Epoch 7/10 | Loss: 0.2587
  Epoch 8/10 | Loss: 0.2442
  Epoch 9/10 | Loss: 0.2347
  Epoch 10/10 | Loss: 0.2533
Precisión Class-IL tras Tarea 0: 92.10%

--- ENTRENANDO TAREA 1 ---
  Epoch 1/10 | Loss: 0.6615
  Epoch 2/10 | Loss: 0.5279
  Epoch 3/10 | Loss: 0.5058
  Epoch 4/10 | Loss: 0.4810
  Epoch 5/10 | Loss: 0.4678
  Epoch 6/10 | Loss: 0.4557
  Epoch 7/10 | Loss: 0.4532
  Epoch 8/10 | Loss: 0.4388
  Epoch 9/10 | Loss: 0.4395
  Epoch 10/10 | Loss: 0.4238
Precisión Class-IL tras Tarea 1: 40.62%

--- ENTRENANDO TAREA 2 ---
  Epoch 1/10 | Loss: 1.0106
  Epoch 2/10 | Loss: 0.5755
  Epoch 3/10 | Loss: 0.5316
  Epoch 4/10 | Loss: 0.5100
  Epoch 5/10 | Loss: 0.4831
  Epoch 6/10 | Loss: 0.4585
  Epoch 7/10 | Loss: 0.4526
  Epoch 8/10 | Loss: 0.4242
  Epoch 9/10 | Loss: 0.4098
  Epoch 10/10 | Loss: 